# 03. REVISAO DOS ENDERECOS COM IA

ESTE NOTEBOOK PADRONIZA O ENDERECO E O BAIRRO DE CADA ACIDENTE USANDO O DEEPSEEK.

DEPENDE DO NOTEBOOK **01**, QUE CRIA A TABELA `acidentes`. E PRE-REQUISITO DO NOTEBOOK **04**.

## POR QUE ESTA ETAPA EXISTE

O RENAEST GRAVA O ENDERECO COMO TEXTO LIVRE, DIGITADO POR QUEM ATENDEU A OCORRENCIA. O RESULTADO E ALGO ASSIM:

```text
AV V-8, 1234 - CEP 74952-160, Aparecida-GO
R. INDEPENDENCIA PROXIMO AO MERCADO
AVN. INDEPENDENCIA KM 3
```

AS TRES PRIMEIRAS COISAS SAO IMPOSSIVEIS DE AGRUPAR. AS DUAS ULTIMAS SAO A MESMA AVENIDA ESCRITA DE DOIS JEITOS.

A IA REDUZ TUDO ISSO A `Avenida V-8` E `Avenida Independencia`, QUE E O QUE O NOTEBOOK 04 CONSEGUE PROCURAR NA TABELA DE VIAS.

## A IDEIA CENTRAL, EM SEIS PASSOS

1. COPIAR AS LINHAS DO RECORTE ESCOLHIDO PARA A TABELA DE RESULTADO.
2. PEGAR OS PARES DISTINTOS DE ENDERECO E BAIRRO QUE AINDA NAO FORAM REVISADOS.
3. SEPARAR O QUE JA ESTA NO DICIONARIO DO QUE E INEDITO.
4. PADRONIZAR SO OS INEDITOS NA IA, EM LOTES.
5. ESPALHAR O RESULTADO PARA TODAS AS LINHAS QUE TEM AQUELE PAR EXATO.
6. REGISTRAR QUANTO FOI FEITO, PARA PODER RETOMAR DEPOIS.

O DICIONARIO DO PASSO 3 E O QUE FAZ ISSO SER BARATO. UM ENDERECO QUE APARECE EM 300 ACIDENTES E ENVIADO A IA **UMA VEZ SO**.

## PROCESSA UM MUNICIPIO POR EXECUCAO

ESCOLHA O ANO E O MUNICIPIO NA ETAPA 1. PARA PROCESSAR OUTRO, TROQUE OS VALORES E EXECUTE DE NOVO. O QUE JA FOI FEITO NAO E REFEITO.

## 1. IMPORTACAO DAS BIBLIOTECAS E CONFIGURACOES

**VOCE NAO EDITA ESTA CELULA.** TODOS OS VALORES ABAIXO VEM DO `parametros.py`, QUE E O UNICO ARQUIVO DE ESCOLHAS DO PROJETO. A CELULA SO DA NOMES CURTOS AOS VALORES, PARA O RESTO DO NOTEBOOK LER.

EXPLICACAO DOS VALORES:

- `ANO` E `CODIGO_IBGE`: O RECORTE QUE SERA PROCESSADO. TIRE ESSES VALORES DA ULTIMA ETAPA DO NOTEBOOK 01.
- `TAMANHO_LOTE`: QUANTOS PARES DISTINTOS VAO EM CADA CHAMADA A IA. LOTE MAIOR GASTA MENOS CHAMADAS, MAS AUMENTA A CHANCE DE A RESPOSTA VIR TRUNCADA -- E RESPOSTA TRUNCADA FAZ O LOTE INTEIRO FALHAR E FICAR PENDENTE, DE PROPOSITO. A ETAPA 3 EXPLICA POR QUE.
- `MAX_TOKENS_RESPOSTA`: TETO DE TOKENS DA RESPOSTA. E O QUE EVITA O TRUNCAMENTO ACIMA.
- `PENSAMENTO_LIGADO`: SE O MODELO RACIOCINA ANTES DE RESPONDER. VEM DESLIGADO, PORQUE AQUI NAO MUDOU O RESULTADO E CUSTAVA DEZ VEZES MAIS TOKENS DE SAIDA.
- `MAX_CONCORRENTES`: QUANTAS CHAMADAS PODEM ESTAR EM VOO AO MESMO TEMPO.
- `DELAY_ENTRE_LOTES`: SEGUNDOS MINIMOS ENTRE O INICIO DE UMA CHAMADA E O INICIO DA SEGUINTE. E O QUE RESPEITA A COTA DA SUA CONTA. O VALOR ERA 6, DIMENSIONADO PARA A COTA GRATUITA DO GEMINI; NO DEEPSEEK ESTA 1. SE APARECER ERRO DE COTA NA ETAPA 6, AUMENTE.
- `TENTATIVAS`: QUANTAS VEZES UM LOTE E TENTADO ANTES DE FICAR PENDENTE PARA A PROXIMA EXECUCAO.

SOBRE A CONCORRENCIA: QUATRO CONCORRENTES COM UM SEGUNDO DE ESPERA NAO SIGNIFICA QUATRO CHAMADAS POR SEGUNDO. SIGNIFICA UMA CHAMADA NOVA POR SEGUNDO, COM ATE QUATRO EM VOO AO MESMO TEMPO. SE CADA CHAMADA DEMORA 20 SEGUNDOS, HAVERA MESMO QUATRO ACONTECENDO JUNTAS.

In [1]:
# JSON MONTA A ENTRADA E LE A RESPOSTA DA IA.
import json

# SQLITE3 E O BANCO LOCAL DO PROJETO.
import sqlite3

# SYS E USADO PARA GARANTIR QUE O PYTHON ENCONTRE O ARQUIVO config.py.
import sys

# THREADING FORNECE A TRAVA QUE CONTROLA O RITMO DAS CHAMADAS.
import threading

# TIME E USADO PARA AS PAUSAS ENTRE CHAMADAS.
import time

# ESTAS DUAS FUNCOES DISPARAM VARIOS LOTES AO MESMO TEMPO.
from concurrent.futures import ThreadPoolExecutor, as_completed

# DATETIME REGISTRA QUANDO CADA COISA FOI FEITA.
from datetime import datetime, timezone

# PATH AJUDA A TRABALHAR COM CAMINHOS DE ARQUIVO.
from pathlib import Path

# PANDAS E USADO SO PARA MOSTRAR TABELAS DE CONFERENCIA.
import pandas as pd

# CLIENTE DA API. O DEEPSEEK FALA O PROTOCOLO DA OPENAI, ENTAO A BIBLIOTECA
# INSTALADA E A `openai` MESMO QUE O MODELO SEJA DEEPSEEK.
from openai import OpenAI

# GARANTE QUE A PASTA projeto_v2 ESTEJA NO CAMINHO DE IMPORTACAO,
# INDEPENDENTE DE ONDE O JUPYTER FOI ABERTO.
for _candidata in (Path.cwd(), Path.cwd() / "projeto_v2", Path.cwd().parent):
    if (_candidata / "config.py").exists():
        sys.path.insert(0, str(_candidata))
        break

# CONFIGURACAO COMPARTILHADA: CAMINHOS, BANCO, CHAVES E NOMES DE COLUNA.
import config

# O QUE SERA PROCESSADO: MUNICIPIO, ANO E AS DEMAIS ESCOLHAS. TODAS ELAS VIVEM NO
# parametros.py -- E O UNICO ARQUIVO QUE VOCE EDITA ANTES DE RODAR.
import parametros

# =========================
# PARAMETROS DESTA EXECUCAO
# =========================

# ANO E MUNICIPIO QUE SERAO PROCESSADOS. UM MUNICIPIO POR EXECUCAO.
# PARA TROCAR DE RECORTE, EDITE O parametros.py -- NAO ESTA CELULA.
ANO = parametros.ANO
CODIGO_IBGE = parametros.CODIGO_IBGE

# =========================
# CONFIGURACOES DAS CHAMADAS A IA
# =========================

# QUANTOS PARES DISTINTOS POR CHAMADA.
TAMANHO_LOTE = parametros.TAMANHO_LOTE_REVISAO

# TETO DE TOKENS DA RESPOSTA. O MODO JSON DA API TRUNCA A RESPOSTA SEM AVISAR SE
# ESTE VALOR FOR CURTO, E RESPOSTA TRUNCADA FAZ O LOTE FALHAR (VEJA A ETAPA 3).
MAX_TOKENS_RESPOSTA = parametros.MAX_TOKENS_RESPOSTA

# O deepseek-v4-flash RACIOCINA ANTES DE RESPONDER POR PADRAO. AQUI ISSO E DESPERDICIO:
# EM TESTE COM OS MESMOS ENDERECOS, A SAIDA FOI IDENTICA COM E SEM RACIOCINIO, E COM
# ELE LIGADO O GASTO DE TOKENS DE SAIDA FOI CERCA DE DEZ VEZES MAIOR.
# COLOQUE True SE QUISER COMPARAR VOCE MESMO.
PENSAMENTO_LIGADO = parametros.PENSAMENTO_LIGADO

# `extra_body` LEVA PARAMETROS QUE SAO DO DEEPSEEK E NAO EXISTEM NA API DA OPENAI.
CORPO_EXTRA = parametros.CORPO_EXTRA

# QUANTAS CHAMADAS PODEM ESTAR EM VOO AO MESMO TEMPO.
MAX_CONCORRENTES = parametros.MAX_CONCORRENTES

# SEGUNDOS MINIMOS ENTRE O INICIO DE UMA CHAMADA E O INICIO DA SEGUINTE.
# ERA 6 PARA CABER NA COTA GRATUITA DO GEMINI. O DEEPSEEK NAO IMPOE ESSE RITMO.
DELAY_ENTRE_LOTES = parametros.DELAY_ENTRE_LOTES

# QUANTAS VEZES TENTAR UM LOTE ANTES DE DEIXAR PARA A PROXIMA EXECUCAO.
TENTATIVAS = parametros.TENTATIVAS

# SEPARADOR INVISIVEL, USADO SO PARA MONTAR A CHAVE DO DICIONARIO.
# E UM CARACTERE DE CONTROLE QUE NUNCA APARECE EM ENDERECO DE VERDADE.
SEPARADOR_CHAVE = "\u001f"

# MOSTRA A CONFIGURACAO ATIVA PARA CONFERENCIA.
config.resumo()
print()
parametros.resumo()

# SEM CHAVE NAO HA COMO CHAMAR A IA.
if not config.CHAVE_IA:
    raise RuntimeError("DEEPSEEK_API_KEY AUSENTE. DEFINA A CHAVE NO ARQUIVO .env DA RAIZ DO REPOSITORIO.")

BANCO DO V2       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\db\db_main.db
  EXISTE?         : SIM
PASTA TEMP        : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\temp
PASTA CACHE       : C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\cache
CHAVE DA IA       : DEFINIDA
MODELO DA IA      : deepseek-v4-flash
URL DA IA         : https://api.deepseek.com
EMBEDDINGS        : LIGADOS
MODELO EMBEDDINGS : intfloat/multilingual-e5-small

RECORTE: ANO 2024, MUNICIPIO 5201405


## 2. A INSTRUCAO ENVIADA A IA

ESTA E A PARTE MAIS IMPORTANTE DO NOTEBOOK, E A UNICA QUE VOCE PROVAVELMENTE VAI QUERER AJUSTAR COM O TEMPO.

A INSTRUCAO E FIXA E VAI JUNTO EM TODA CHAMADA. ELA PRECISA GARANTIR TRES COISAS:

1. **SAIDA ESTAVEL.** A MESMA ENTRADA TEM QUE PRODUZIR SEMPRE A MESMA SAIDA. POR ISSO A TEMPERATURA E ZERO E A INSTRUCAO INSISTE NISSO.
2. **FORMATO PREVISIVEL.** A RESPOSTA E UM OBJETO JSON COM A CHAVE `resultados`, E DENTRO DELA UM ARRAY DO MESMO TAMANHO E NA MESMA ORDEM DA ENTRADA. E ASSIM QUE CASAMOS CADA RESPOSTA COM O PAR CORRETO.

   O ARRAY VEM DENTRO DE UM OBJETO POR EXIGENCIA DA API: NO MODO JSON DELA, A RESPOSTA E SEMPRE UM OBJETO. A INSTRUCAO TAMBEM PRECISA CONTER A PALAVRA `json` E UM EXEMPLO DA SAIDA -- SEM ISSO A API RECUSA A CHAMADA. NAO REMOVA O EXEMPLO.
3. **NADA DE INVENCAO.** A IA PODE EXPANDIR ABREVIACAO E ARRUMAR CAIXA, MAS NAO PODE TRADUZIR NEM CORRIGIR NOME PROPRIO.

REPARE NA ULTIMA REGRA, SOBRE VALOR AUSENTE. QUANDO O ENDERECO E `S/N`, `NAO INFORMADO` OU SO UM NUMERO, A INSTRUCAO MANDA RESPONDER `null`.

ISSO E UMA RESPOSTA **CORRETA**, NAO UMA FALHA, E O CODIGO PRECISA TRATAR AS DUAS COISAS DE FORMA DIFERENTE. A ETAPA 3 EXPLICA POR QUE ISSO IMPORTA TANTO.

In [2]:
INSTRUCAO = """Voce e um padronizador de enderecos de acidentes de transito do Brasil. A saida deve ser
ESTAVEL e PREVISIVEL: a MESMA entrada produz SEMPRE a MESMA saida, sem variacao nem invencao.
Recebe um JSON array de objetos {"end": <logradouro bruto>, "bairro": <bairro bruto>}.

FORMATO DA SAIDA (obrigatorio):
- Responda APENAS um objeto json com a chave "resultados", cujo valor e um array na
  MESMA ORDEM e MESMO TAMANHO da entrada.
- Cada item do array exatamente: {"end_padronizado": <str|null>, "bairro_padronizado": <str|null>}.
- Nada fora do json: sem markdown, sem comentarios, sem texto extra.
- Exemplo de saida json valida, para uma entrada de dois itens:
  {"resultados": [{"end_padronizado": "Avenida V-8", "bairro_padronizado": "Jardim Tropical"},
                  {"end_padronizado": null, "bairro_padronizado": null}]}

LOGRADOURO - mantenha SOMENTE o tipo + nome do logradouro. REMOVA sempre: o CEP e tudo que
vier depois dele; numero de porta; KM; complementos (bloco, lote, quadra, apartamento);
pontos de referencia ('esquina com', 'proximo a', 'em frente a'); cidade, bairro e UF.
  ex.: 'AV V-8, 1234 - CEP 74952-160, Aparecida-GO' -> 'Avenida V-8'.

REGRAS DO NOME DO LOGRADOURO:
- Expanda o tipo abreviado: AV/AVN->Avenida, R/RU->Rua, TV/TRAV/TRV->Travessa, ROD->Rodovia,
  PC/PCA->Praca, AL->Alameda, ESTR/EST->Estrada, LRG/LGO->Largo, VD->Viaduto, BC->Beco,
  ACS->Acesso, ANEL->Anel Viario.
- Se NAO houver tipo claro, devolva so o nome, SEM inventar um tipo.
- NAO invente, traduza nem 'corrija' nomes proprios: preserve o nome (apenas ajuste a caixa).
- Mantenha numeros que fazem PARTE do nome: 'Avenida 85', 'Rua 7 de Setembro', 'Alameda 21 de Abril'.
- Identificador letra+numero: sempre LETRA-NUMERO, com hifen e em maiuscula
  (ex.: 'v 8'->'V-8', 'a10'->'A-10', 'Rua J17'->'Rua J-17').
- Rodovia federal/estadual: formate como 'Rodovia SIGLA-NUMERO', sigla em maiusculas e com hifen
  (ex.: 'BR 153'->'Rodovia BR-153', 'go-040'->'Rodovia GO-040'). Se a entrada JA trouxer outro
  tipo (ex.: 'Rua MG 03'), mantenha o tipo informado.

BAIRRO - mantenha SOMENTE o nome do bairro (remova CEP e qualquer extra). Expanda abreviacoes
  comuns: JD->Jardim, VL->Vila, PQ->Parque, CJ/CONJ->Conjunto, RES->Residencial, ST/SET->Setor,
  CH/CHAC->Chacara.

CAPITALIZACAO (vale para logradouro e bairro):
- Use Caixa de Titulo (inicial maiuscula em cada palavra) e SEM acentos (saida em ASCII puro),
  para casar com o restante da base ja padronizada.
- Conectivos sempre em minusculo: de, da, do, das, dos, e
  (ex.: 'AV. SETE DE SETEMBRO' -> 'Avenida Sete de Setembro').

VALOR AUSENTE: se o campo for vazio/nulo, ou um marcador sem logradouro/bairro real
('S/N', 'SEM NOME', 'NAO INFORMADO', 'DESCONHECIDO', so um numero ou traco), retorne null nele.

ENTRADA:
"""

print(f"INSTRUCAO CARREGADA: {len(INSTRUCAO)} CARACTERES.")

INSTRUCAO CARREGADA: 2780 CARACTERES.


## 3. FUNCOES AUXILIARES

AQUI ESTAO TODAS AS FUNCOES USADAS PELAS ETAPAS SEGUINTES.

### A CORRECAO MAIS IMPORTANTE DESTE NOTEBOOK

A VERSAO ANTERIOR DESTE CODIGO TRATAVA DUAS COISAS DIFERENTES COMO SE FOSSEM A MESMA:

```text
CASO A: A CHAMADA A IA FALHOU (REDE, COTA, ERRO, RESPOSTA FORA DE FORMA).
        -> NAO SABEMOS A RESPOSTA. O PAR DEVE FICAR PENDENTE E SER TENTADO DE NOVO.

CASO B: A IA RESPONDEU null DE PROPOSITO, PORQUE O ENDERECO ERA 'NAO INFORMADO'.
        -> ESSA E A RESPOSTA CERTA. O PAR ESTA RESOLVIDO E NAO DEVE SER TENTADO DE NOVO.
```

OS DOIS CASOS CHEGAVAM COMO `None` E OS DOIS ERAM DESCARTADOS. O EFEITO ERA RUIM E SILENCIOSO:

- OS ENDERECOS IMPOSSIVEIS NUNCA ENTRAVAM NO DICIONARIO.
- ELES FICAVAM `pendente` PARA SEMPRE.
- A COBERTURA DO MUNICIPIO NUNCA CHEGAVA A `ok`, FICAVA ETERNAMENTE EM `parcial`.
- COMO A COBERTURA NUNCA FECHAVA, A PROTECAO DE RETOMADA NUNCA ATUAVA.
- **TODA EXECUCAO REENVIAVA OS MESMOS ENDERECOS IMPOSSIVEIS A IA, GASTANDO COTA DE NOVO.**

A CORRECAO ESTA EM `padronizar_lote`: ELA DEVOLVE `None` INTEIRO QUANDO A CHAMADA FALHOU, E UMA LISTA QUANDO A CHAMADA DEU CERTO, MESMO QUE ALGUNS ITENS DENTRO DELA SEJAM `null`.

ASSIM O CASO B ENTRA NO DICIONARIO, VIRA `revisao_status = 'ok'` COM O PADRONIZADO VAZIO, E NUNCA MAIS E REENVIADO. O NOTEBOOK 04 SABE O QUE FAZER COM ELE.

### O MESMO CUIDADO, APLICADO A RESPOSTA INCOMPLETA

A FUNCAO `extrair_itens` LEVANTA ERRO QUANDO A RESPOSTA VEM COM MENOS ITENS DO QUE O LOTE ENVIADO, OU FORA DA FORMA ESPERADA.

PARECE RIGOR EXCESSIVO, MAS E A MESMA ARMADILHA DE NOVO. NA VERSAO ANTERIOR, ITEM QUE FALTAVA VIRAVA PAR VAZIO -- E PAR VAZIO AQUI SIGNIFICA `ENDERECO SEM LOGRADOURO`, QUE E GRAVADO COMO DECISAO CONCLUIDA E **NUNCA MAIS REVISTO**.

UMA UNICA RESPOSTA TRUNCADA APAGARIA ENDERECOS BONS PARA SEMPRE, SEM NADA NA TABELA INDICANDO ISSO. ENTAO O LOTE INTEIRO FALHA E VOLTA NA PROXIMA EXECUCAO. E BARATO: O QUE JA FOI RESOLVIDO ESTA NO DICIONARIO E NAO E REENVIADO.

In [3]:
def agora():
    """HORARIO ATUAL EM TEXTO, PARA REGISTRAR QUANDO ALGO FOI FEITO."""
    return datetime.now(timezone.utc).isoformat()


def chave(end, bairro):
    """MONTA A CHAVE UNICA DE UM PAR. UM VALOR NULO VIRA TEXTO VAZIO."""
    return f"{end or ''}{SEPARADOR_CHAVE}{bairro or ''}"


def conectar():
    """ABRE A CONEXAO COM O BANCO DO PROJETO."""
    return sqlite3.connect(str(config.BANCO))


def colunas_do_acidentes(conn):
    """LISTA AS COLUNAS DA TABELA acidentes, PARA COPIAR TODAS ELAS."""
    return [(linha[1], linha[2] or "TEXT") for linha in conn.execute("PRAGMA table_info(acidentes)")]


def validar_tabela_acidentes(conn):
    """CONFERE QUE A TABELA acidentes EXISTE E TEM AS COLUNAS QUE PRECISAMOS.

    FALHA AQUI, COM MENSAGEM CLARA, EM VEZ DE ESTOURAR UM ERRO DE SQL NO MEIO
    DO PROCESSAMENTO.
    """
    colunas = {nome for nome, _ in colunas_do_acidentes(conn)}

    if not colunas:
        raise RuntimeError(
            "A TABELA 'acidentes' NAO EXISTE NO BANCO.\n"
            "RODE O NOTEBOOK 01_ingestao_renaest.ipynb PRIMEIRO."
        )

    faltando = [c for c in config.COLUNAS_OBRIGATORIAS_ACIDENTES if c not in colunas]
    if faltando:
        raise RuntimeError(
            f"A TABELA 'acidentes' NAO TEM AS COLUNAS {faltando}.\n"
            "VEJA A ETAPA 6 DO NOTEBOOK 01 E CORRIJA AS CONSTANTES COLUNA_* NO config.py."
        )

    print(f"TABELA 'acidentes' OK: {len(colunas)} COLUNAS, TODAS AS OBRIGATORIAS PRESENTES.")


def criar_tabelas(conn):
    """CRIA AS TRES TABELAS DESTE NOTEBOOK, SE AINDA NAO EXISTIREM."""
    colunas = colunas_do_acidentes(conn)

    # A TABELA DE RESULTADO E UMA COPIA DAS COLUNAS DO acidentes MAIS AS NOVAS.
    definicoes = []
    for nome, tipo in colunas:
        # O IDENTIFICADOR DO ACIDENTE VIRA CHAVE PRIMARIA, O QUE IMPEDE DUPLICATA.
        if nome == config.COLUNA_ID_ACIDENTE:
            definicoes.append(f"{nome} {tipo} PRIMARY KEY")
        else:
            definicoes.append(f"{nome} {tipo}")

    definicoes += [
        "end_acidente_padronizado TEXT",
        "bairro_acidente_padronizado TEXT",
        "revisao_status TEXT",     # 'pendente' OU 'ok'
        "revisao_modelo TEXT",
        "revisao_em TEXT",
    ]

    corpo = ",\n  ".join(definicoes)
    conn.execute(f"CREATE TABLE IF NOT EXISTS acidentes_revisado (\n  {corpo}\n)")

    # INDICE QUE ACELERA AS CONSULTAS POR RECORTE.
    conn.execute(
        f"CREATE INDEX IF NOT EXISTS idx_rev_slice "
        f"ON acidentes_revisado({config.COLUNA_ANO}, {config.COLUNA_MUNICIPIO}, revisao_status)"
    )

    # DICIONARIO REUSAVEL: UMA LINHA POR PAR DISTINTO JA DECIDIDO.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS padronizacao (
            chave TEXT PRIMARY KEY,
            end_raw TEXT,
            bairro_raw TEXT,
            end_padronizado TEXT,
            bairro_padronizado TEXT,
            sem_endereco INTEGER DEFAULT 0,
            modelo TEXT,
            criado_em TEXT
        )
    """)

    # A COLUNA sem_endereco FOI ACRESCENTADA DEPOIS. GARANTE QUE ELA EXISTA
    # EM BANCOS CRIADOS POR UMA VERSAO ANTERIOR DESTE NOTEBOOK.
    colunas_dic = {linha[1] for linha in conn.execute("PRAGMA table_info(padronizacao)")}
    if "sem_endereco" not in colunas_dic:
        conn.execute("ALTER TABLE padronizacao ADD COLUMN sem_endereco INTEGER DEFAULT 0")

    # MAPA DE PROGRESSO POR ANO E MUNICIPIO.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS revisao_cobertura (
            ano TEXT,
            codigo_ibge TEXT,
            status TEXT,
            total_linhas INTEGER,
            distintos INTEGER,
            novos_ia INTEGER,
            reaproveitados INTEGER,
            iniciado_em TEXT,
            concluido_em TEXT,
            PRIMARY KEY (ano, codigo_ibge)
        )
    """)

    conn.commit()


def extrair_itens(texto, tamanho_esperado):
    """TIRA O ARRAY DE RESULTADOS DA RESPOSTA E CONFERE QUE ELE VEIO COMPLETO.

    LEVANTA ERRO SEMPRE QUE A RESPOSTA NAO SERVE. QUEM CHAMA TRATA ISSO COMO
    CHAMADA QUE FALHOU: O LOTE FICA PENDENTE E E TENTADO DE NOVO.

    ISSO E DELIBERADO, E E A DIFERENCA MAIS IMPORTANTE EM RELACAO A VERSAO COM
    GEMINI. LA, UMA RESPOSTA CURTA OU FORA DE FORMA VIRAVA ITEM VAZIO -- E ITEM
    VAZIO, NESTE PROJETO, SIGNIFICA 'DECISAO CONCLUIDA SEM RESULTADO', QUE E
    GRAVADA NO DICIONARIO E NUNCA MAIS REVISTA. UMA RESPOSTA TRUNCADA APAGARIA
    ENDERECOS PARA SEMPRE, EM SILENCIO. AGORA ELA FALHA ALTO.

    O MODO JSON DESTA API DEVOLVE UM OBJETO, NAO UM ARRAY. POR ISSO A INSTRUCAO
    PEDE O ARRAY DENTRO DA CHAVE 'resultados'.
    """
    dados = json.loads(texto)

    if isinstance(dados, dict):
        itens = dados.get("resultados")

        # SE O MODELO INVENTOU OUTRO NOME DE CHAVE, PEGA O PRIMEIRO ARRAY QUE ACHAR.
        if not isinstance(itens, list):
            itens = next((v for v in dados.values() if isinstance(v, list)), None)
    else:
        # ARRAY NA RAIZ TAMBEM E ACEITO, CASO A API MUDE DE COMPORTAMENTO.
        itens = dados

    if not isinstance(itens, list):
        raise ValueError(f"RESPOSTA SEM ARRAY DE RESULTADOS: {texto[:200]}")

    if len(itens) != tamanho_esperado:
        raise ValueError(
            f"RESPOSTA COM {len(itens)} ITENS, ESPERADOS {tamanho_esperado}. "
            f"SE REPETIR, DIMINUA TAMANHO_LOTE OU AUMENTE MAX_TOKENS_RESPOSTA."
        )

    if not all(isinstance(item, dict) for item in itens):
        raise ValueError("RESPOSTA COM ITEM QUE NAO E OBJETO JSON.")

    return itens


def pedir_a_ia(client, modelo, lote):
    """MANDA UM LOTE DE PARES E DEVOLVE A LISTA DE RESPOSTAS, NA MESMA ORDEM.

    QUALQUER PROBLEMA COM A RESPOSTA VIRA EXCECAO, TRATADA POR padronizar_lote.
    """
    # MONTA A ENTRADA: UM OBJETO POR PAR, SO COM O QUE A IA PRECISA VER.
    entrada = [{"end": p["end"], "bairro": p["bairro"]} for p in lote]

    resposta = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user",
                   "content": INSTRUCAO + json.dumps(entrada, ensure_ascii=False)}],
        response_format={"type": "json_object"},
        temperature=0,   # ZERO PARA A SAIDA SER A MAIS ESTAVEL POSSIVEL.
        max_tokens=MAX_TOKENS_RESPOSTA,
        extra_body=CORPO_EXTRA,
    )

    # CONFERE A FORMA E O TAMANHO ANTES DE OLHAR O CONTEUDO.
    itens = extrair_itens(resposta.choices[0].message.content, len(lote))

    # ALINHA A RESPOSTA COM A ENTRADA, ITEM A ITEM, NA ORDEM.
    return [(item.get("end_padronizado"), item.get("bairro_padronizado")) for item in itens]


def padronizar_lote(client, modelo, lote):
    """CHAMA A IA COM RETENTATIVA. DEVOLVE None SE A CHAMADA FALHOU DE VEZ.

    ESTA E A FUNCAO QUE SEPARA OS DOIS CASOS EXPLICADOS ACIMA:

    - DEVOLVE UMA LISTA  -> A CHAMADA DEU CERTO. UM null DENTRO DELA E RESPOSTA VALIDA.
    - DEVOLVE None       -> A CHAMADA FALHOU. NAO SABEMOS NADA SOBRE ESSES PARES.

    A VERSAO ANTERIOR DEVOLVIA UMA LISTA DE PARES VAZIOS NOS DOIS CASOS, E POR
    ISSO CONFUNDIA 'ENDERECO IMPOSSIVEL' COM 'CHAMADA QUE FALHOU'.

    RESPOSTA INCOMPLETA OU FORA DE FORMA CAI NO SEGUNDO CASO, POR DECISAO DE
    extrair_itens.
    """
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            return pedir_a_ia(client, modelo, lote)
        except Exception as erro:
            if tentativa == TENTATIVAS:
                print(f"\n[REVISAO] LOTE FALHOU APOS {TENTATIVAS} TENTATIVAS: {erro}")
                print("[REVISAO] ESSES PARES FICAM PENDENTES PARA A PROXIMA EXECUCAO.")
                return None
            # ESPERA UM POUCO MAIS A CADA TENTATIVA.
            time.sleep(2 * tentativa)


def salvar_no_dicionario(conn, modelo, lote, resultados):
    """GRAVA NO DICIONARIO O RESULTADO DE UM LOTE QUE DEU CERTO.

    UM PAR COM end_padronizado NULO E GRAVADO ASSIM MESMO, MARCADO COM
    sem_endereco = 1. ELE ESTA RESOLVIDO: O ENDERECO ORIGINAL NAO TINHA
    LOGRADOURO NENHUM E NUNCA VAI TER.
    """
    # CHAMADA QUE FALHOU NAO GRAVA NADA: OS PARES CONTINUAM PENDENTES.
    if resultados is None:
        return 0

    gravados = 0
    for par, (end_padr, bairro_padr) in zip(lote, resultados):
        # 1 QUANDO A IA DISSE QUE NAO HA LOGRADOURO APROVEITAVEL NESSE ENDERECO.
        sem_endereco = 1 if end_padr is None else 0

        conn.execute(
            "INSERT OR REPLACE INTO padronizacao "
            "(chave, end_raw, bairro_raw, end_padronizado, bairro_padronizado, "
            " sem_endereco, modelo, criado_em) "
            "VALUES (?,?,?,?,?,?,?,?)",
            (chave(par["end"], par["bairro"]), par["end"], par["bairro"],
             end_padr, bairro_padr, sem_endereco, modelo, agora()),
        )
        gravados += 1

    conn.commit()
    return gravados


def padronizar_novos(conn, client, modelo, novos):
    """PADRONIZA OS PARES INEDITOS: VARIOS LOTES AO MESMO TEMPO, COM RITMO CONTROLADO."""
    # QUEBRA A LISTA DE PARES EM LOTES DO TAMANHO CONFIGURADO.
    lotes = [novos[i:i + TAMANHO_LOTE] for i in range(0, len(novos), TAMANHO_LOTE)]

    # A TRAVA GARANTE QUE DUAS CHAMADAS NAO COMECEM COLADAS UMA NA OUTRA.
    trava = threading.Lock()
    proximo_inicio = [0.0]

    def chamar(lote):
        # SO O AGENDAMENTO FICA DENTRO DA TRAVA. A CHAMADA EM SI ACONTECE FORA,
        # ENTAO VARIAS PODEM ESTAR EM VOO AO MESMO TEMPO.
        with trava:
            espera = proximo_inicio[0] - time.time()
            if espera > 0:
                time.sleep(espera)
            proximo_inicio[0] = time.time() + DELAY_ENTRE_LOTES

        return lote, padronizar_lote(client, modelo, lote)

    resolvidos = 0
    falhados = 0

    # DISPARA OS LOTES EM PARALELO. A GRAVACAO ACONTECE NA THREAD PRINCIPAL,
    # CONFORME CADA LOTE VOLTA, PORQUE A CONEXAO SQLITE NAO E COMPARTILHAVEL.
    with ThreadPoolExecutor(max_workers=MAX_CONCORRENTES) as pool:
        tarefas = [pool.submit(chamar, lote) for lote in lotes]

        for indice, tarefa in enumerate(as_completed(tarefas), start=1):
            lote, resultados = tarefa.result()

            if resultados is None:
                falhados += len(lote)
            else:
                resolvidos += salvar_no_dicionario(conn, modelo, lote, resultados)

            print(f"\rLOTE {indice}/{len(lotes)}  |  RESOLVIDOS: {resolvidos}  |  "
                  f"PENDENTES POR FALHA: {falhados}", end="")

    print()
    return resolvidos, falhados


def copiar_linhas(conn, ano, ibge):
    """PASSO 1: COPIA AS LINHAS DO RECORTE PARA acidentes_revisado, SEM DUPLICAR.

    O 'INSERT OR IGNORE' IMPEDE DUPLICATA PORQUE O IDENTIFICADOR DO ACIDENTE E
    CHAVE PRIMARIA. ATENCAO AO EFEITO COLATERAL: SE VOCE REEXECUTAR O NOTEBOOK 01
    E O DADO DE ORIGEM TIVER MUDADO, A LINHA JA COPIADA **NAO** E ATUALIZADA.
    ELA GUARDA A VERSAO QUE FOI REVISADA NA EPOCA.
    """
    colunas = ", ".join(nome for nome, _ in colunas_do_acidentes(conn))

    antes = conn.execute(
        f"SELECT count(*) FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=?", (ano, ibge)
    ).fetchone()[0]

    conn.execute(
        f"INSERT OR IGNORE INTO acidentes_revisado ({colunas}, revisao_status) "
        f"SELECT {colunas}, 'pendente' FROM acidentes "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=?",
        (ano, ibge),
    )
    conn.commit()

    depois = conn.execute(
        f"SELECT count(*) FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=?", (ano, ibge)
    ).fetchone()[0]

    return depois - antes, depois


def pares_pendentes(conn, ano, ibge):
    """PASSO 2: PEGA OS PARES DISTINTOS DE ENDERECO E BAIRRO AINDA PENDENTES."""
    return conn.execute(
        f"SELECT DISTINCT {config.COLUNA_ENDERECO}, {config.COLUNA_BAIRRO} "
        f"FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? AND revisao_status='pendente'",
        (ano, ibge),
    ).fetchall()


def separar_novos(conn, pares):
    """PASSO 3: SEPARA OS PARES QUE O DICIONARIO JA CONHECE DOS QUE SAO INEDITOS."""
    novos = []
    for end, bairro in pares:
        ja_conhecido = conn.execute(
            "SELECT 1 FROM padronizacao WHERE chave=?", (chave(end, bairro),)
        ).fetchone()

        if not ja_conhecido:
            novos.append({"end": end, "bairro": bairro})

    return novos


def espalhar_resultado(conn, modelo, ano, ibge):
    """PASSO 5: COPIA A VERSAO PADRONIZADA PARA TODAS AS LINHAS DAQUELE PAR.

    UM PAR QUE A IA MARCOU COMO SEM ENDERECO TAMBEM E ESPALHADO. ELE VIRA
    revisao_status = 'ok' COM O PADRONIZADO NULO, PORQUE A REVISAO DELE ESTA
    CONCLUIDA: NAO HA LOGRADOURO PARA PADRONIZAR.

    O 'IS' NO LUGAR DE '=' E NECESSARIO PORQUE O BAIRRO PODE SER NULO, E EM SQL
    NULO NUNCA E IGUAL A NULO.
    """
    resolvidos = 0
    sem_endereco = 0

    for end, bairro in pares_pendentes(conn, ano, ibge):
        decisao = conn.execute(
            "SELECT end_padronizado, bairro_padronizado, sem_endereco "
            "FROM padronizacao WHERE chave=?",
            (chave(end, bairro),),
        ).fetchone()

        # PAR AINDA NAO DECIDIDO: FICA PENDENTE PARA A PROXIMA EXECUCAO.
        if decisao is None:
            continue

        end_padr, bairro_padr, marca_sem_endereco = decisao

        cursor = conn.execute(
            f"UPDATE acidentes_revisado "
            f"SET end_acidente_padronizado=?, bairro_acidente_padronizado=?, "
            f"    revisao_status='ok', revisao_modelo=?, revisao_em=? "
            f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? "
            f"  AND revisao_status='pendente' "
            f"  AND {config.COLUNA_ENDERECO} IS ? AND {config.COLUNA_BAIRRO} IS ?",
            (end_padr, bairro_padr, modelo, agora(), ano, ibge, end, bairro),
        )

        resolvidos += cursor.rowcount
        if marca_sem_endereco:
            sem_endereco += cursor.rowcount

    conn.commit()
    return resolvidos, sem_endereco


def registrar_cobertura(conn, ano, ibge, distintos, novos):
    """PASSO 6: ANOTA NA COBERTURA QUANTO FOI FEITO NESTE RECORTE.

    OS CONTADORES SAO SOMADOS AO QUE JA EXISTIA. A VERSAO ANTERIOR SUBSTITUIA,
    ENTAO UMA EXECUCAO DE RETOMADA, QUE PROCESSA POUCOS PARES, SOBRESCREVIA O
    TOTAL COM UM NUMERO MENOR E O RELATORIO FICAVA ERRADO.
    """
    total = conn.execute(
        f"SELECT count(*) FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=?", (ano, ibge)
    ).fetchone()[0]

    pendentes = conn.execute(
        f"SELECT count(*) FROM acidentes_revisado "
        f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? AND revisao_status='pendente'",
        (ano, ibge),
    ).fetchone()[0]

    # LE OS CONTADORES DE UMA EXECUCAO ANTERIOR, SE HOUVER.
    anterior = conn.execute(
        "SELECT distintos, novos_ia FROM revisao_cobertura WHERE ano=? AND codigo_ibge=?",
        (ano, ibge),
    ).fetchone()
    distintos_antes, novos_antes = anterior if anterior else (0, 0)

    distintos_total = (distintos_antes or 0) + distintos
    novos_total = (novos_antes or 0) + novos

    # SO E 'ok' QUANDO NAO SOBROU NENHUMA LINHA PENDENTE.
    status = "ok" if pendentes == 0 else "parcial"

    conn.execute(
        "INSERT OR REPLACE INTO revisao_cobertura "
        "(ano, codigo_ibge, status, total_linhas, distintos, novos_ia, reaproveitados, concluido_em) "
        "VALUES (?,?,?,?,?,?,?,?)",
        (ano, ibge, status, total, distintos_total, novos_total,
         distintos_total - novos_total, agora()),
    )
    conn.commit()

    return {"total": total, "distintos": distintos_total, "novos_ia": novos_total,
            "reaproveitados": distintos_total - novos_total,
            "pendentes": pendentes, "status": status}


print("FUNCOES AUXILIARES CARREGADAS.")

FUNCOES AUXILIARES CARREGADAS.


## 4. PREPARAR O BANCO E CONFERIR O RECORTE

ESTA ETAPA FAZ TRES CONFERENCIAS ANTES DE GASTAR QUALQUER CHAMADA DE IA:

1. A TABELA `acidentes` EXISTE E TEM AS COLUNAS QUE PRECISAMOS.
2. EXISTE DADO PARA O ANO E O MUNICIPIO ESCOLHIDOS.
3. ESSE RECORTE JA FOI CONCLUIDO ANTES?

SE O RECORTE JA ESTIVER MARCADO COMO `ok` NA COBERTURA, O NOTEBOOK AVISA E VOCE PODE PARAR AQUI. NAO HA NADA A REFAZER.

In [4]:
# OS VALORES SAO GUARDADOS COMO TEXTO NO BANCO, ENTAO CONVERTEMOS AQUI.
ano = str(ANO)
ibge = str(CODIGO_IBGE)

# CRIA O CLIENTE DA IA UMA VEZ SO. A URL E O QUE APONTA O CLIENTE DA OPENAI
# PARA O DEEPSEEK; O RESTO DO CODIGO NAO SABE QUAL PROVEDOR ESTA ATENDENDO.
client = OpenAI(api_key=config.CHAVE_IA, base_url=config.URL_BASE_IA)
modelo = config.MODELO_IA

conn = conectar()

# CONFERE QUE A TABELA DE ORIGEM ESTA COMO ESPERAMOS.
validar_tabela_acidentes(conn)

# CRIA AS TABELAS DESTE NOTEBOOK.
criar_tabelas(conn)
print("TABELAS acidentes_revisado, padronizacao E revisao_cobertura PRONTAS.")
print()

# CONFERE QUE EXISTE DADO BRUTO PARA ESSE RECORTE.
tem_dado = conn.execute(
    f"SELECT count(*) FROM acidentes "
    f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=?", (ano, ibge)
).fetchone()[0]

if tem_dado == 0:
    raise RuntimeError(
        f"NENHUM ACIDENTE EM 'acidentes' PARA ANO={ano}, MUNICIPIO={ibge}.\n"
        "CONFIRA OS RECORTES DISPONIVEIS NA ULTIMA ETAPA DO NOTEBOOK 01."
    )

print(f"ACIDENTES NO RECORTE: {tem_dado}")

# VERIFICA SE ESSE RECORTE JA FOI CONCLUIDO EM UMA EXECUCAO ANTERIOR.
cobertura = conn.execute(
    "SELECT status, total_linhas, distintos, novos_ia FROM revisao_cobertura "
    "WHERE ano=? AND codigo_ibge=?", (ano, ibge)
).fetchone()

RECORTE_JA_CONCLUIDO = bool(cobertura and cobertura[0] == "ok")

print()
if RECORTE_JA_CONCLUIDO:
    print("ESTE RECORTE JA FOI CONCLUIDO EM UMA EXECUCAO ANTERIOR.")
    print(f"   LINHAS         : {cobertura[1]}")
    print(f"   PARES DISTINTOS: {cobertura[2]}")
    print(f"   ENVIADOS A IA  : {cobertura[3]}")
    print()
    print("AS ETAPAS 5 A 7 SERAO PULADAS. VA PARA A ETAPA 8.")
elif cobertura:
    print(f"ESTE RECORTE ESTA PARCIAL. A EXECUCAO VAI CONTINUAR DE ONDE PAROU.")
else:
    print("RECORTE INEDITO. A EXECUCAO VAI PROCESSAR TUDO.")

TABELA 'acidentes' OK: 35 COLUNAS, TODAS AS OBRIGATORIAS PRESENTES.
TABELAS acidentes_revisado, padronizacao E revisao_cobertura PRONTAS.

ACIDENTES NO RECORTE: 8865

RECORTE INEDITO. A EXECUCAO VAI PROCESSAR TUDO.


## 5. PASSOS 1 A 3: COPIAR, LISTAR E SEPARAR

TRES PASSOS BARATOS, TODOS DENTRO DO BANCO, SEM NENHUMA CHAMADA DE IA.

**PASSO 1** COPIA AS LINHAS DO RECORTE PARA `acidentes_revisado`, MARCADAS COMO `pendente`.

**PASSO 2** PEGA OS PARES **DISTINTOS** DE ENDERECO E BAIRRO. AQUI ESTA A PRIMEIRA GRANDE ECONOMIA: UM MUNICIPIO PODE TER 20 MIL ACIDENTES E APENAS 3 MIL ENDERECOS DIFERENTES.

**PASSO 3** CONSULTA O DICIONARIO E SEPARA O QUE JA FOI DECIDIDO ANTES. ESSA E A SEGUNDA ECONOMIA, E ELA CRESCE A CADA MUNICIPIO PROCESSADO: ENDERECOS COMUNS COMO `Rodovia BR-153` JA ESTARAO LA.

A SAIDA DESTA CELULA MOSTRA EXATAMENTE QUANTAS CHAMADAS DE IA VOCE VAI GASTAR.

In [5]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
    pares = []
    novos = []
else:
    # PASSO 1: COPIA AS LINHAS DO RECORTE.
    copiadas, total_no_recorte = copiar_linhas(conn, ano, ibge)
    print(f"PASSO 1  LINHAS COPIADAS AGORA : {copiadas}")
    print(f"         LINHAS NO RECORTE     : {total_no_recorte}")
    print()

    # PASSO 2: PEGA OS PARES DISTINTOS AINDA PENDENTES.
    pares = pares_pendentes(conn, ano, ibge)
    print(f"PASSO 2  PARES DISTINTOS PENDENTES: {len(pares)}")
    if total_no_recorte:
        print(f"         ISSO E {len(pares) / total_no_recorte * 100:.1f}% DO NUMERO DE LINHAS.")
    print()

    # PASSO 3: SEPARA OS QUE O DICIONARIO JA CONHECE.
    novos = separar_novos(conn, pares)
    reaproveitados = len(pares) - len(novos)
    print(f"PASSO 3  JA NO DICIONARIO         : {reaproveitados}")
    print(f"         INEDITOS, VAO PARA A IA  : {len(novos)}")
    print()

    if novos:
        lotes_previstos = (len(novos) + TAMANHO_LOTE - 1) // TAMANHO_LOTE
        tempo_estimado = lotes_previstos * DELAY_ENTRE_LOTES / 60
        print(f"SERAO {lotes_previstos} CHAMADAS A IA.")
        print(f"TEMPO MINIMO ESTIMADO: {tempo_estimado:.1f} MINUTOS (SO PELO RITMO IMPOSTO).")
    else:
        print("NENHUMA CHAMADA A IA SERA NECESSARIA. TUDO VEM DO DICIONARIO.")

    # AMOSTRA DO QUE SERA ENVIADO, PARA VOCE VER O TIPO DE TEXTO QUE CHEGA.
    if novos:
        print()
        print("AMOSTRA DO QUE SERA ENVIADO A IA:")
        for p in novos[:5]:
            print(f"   end={p['end']!r}  bairro={p['bairro']!r}")

PASSO 1  LINHAS COPIADAS AGORA : 8865
         LINHAS NO RECORTE     : 8865

PASSO 2  PARES DISTINTOS PENDENTES: 5249
         ISSO E 59.2% DO NUMERO DE LINHAS.

PASSO 3  JA NO DICIONARIO         : 0
         INEDITOS, VAO PARA A IA  : 5249

SERAO 175 CHAMADAS A IA.
TEMPO MINIMO ESTIMADO: 2.9 MINUTOS (SO PELO RITMO IMPOSTO).

AMOSTRA DO QUE SERA ENVIADO A IA:
   end='ALAMEDA DO ALMEIDA  CEP 74915-020'  bairro='SETOR JARDIM LUZ'
   end='RUA J 32  CEP 74950-010'  bairro='MANSOES PARAISO'
   end='RUA R 4  CEP 74960-610'  bairro='PARQUE IBIRAPUERA'
   end='RUA TAPAJOS  CEP 74905-700'  bairro='VILA BRASILIA'
   end='RUA JOSE CANDIDO QUEIROZ  CEP 74980-070'  bairro='SETOR CENTRAL'


## 6. PASSO 4: PADRONIZAR OS INEDITOS NO DEEPSEEK

ESTA E A ETAPA DEMORADA E A UNICA QUE CUSTA DINHEIRO.

OS LOTES SAO DISPARADOS EM PARALELO, MAS COM RITMO CONTROLADO: UMA CHAMADA NOVA A CADA `DELAY_ENTRE_LOTES` SEGUNDOS, COM ATE `MAX_CONCORRENTES` EM VOO.

O CONTADOR MOSTRA DUAS COISAS SEPARADAS:

- **RESOLVIDOS**: PARES QUE A IA DECIDIU E QUE JA ESTAO NO DICIONARIO. INCLUI OS QUE ELA MARCOU COMO SEM ENDERECO APROVEITAVEL, QUE TAMBEM ESTAO RESOLVIDOS.
- **PENDENTES POR FALHA**: PARES DE LOTES CUJA CHAMADA NAO COMPLETOU. ESSES CONTINUAM PENDENTES E SERAO TENTADOS NA PROXIMA EXECUCAO.

SE A SEGUNDA CONTAGEM VIER ALTA, OLHE A MENSAGEM DE ERRO IMPRESSA. COTA PEDE `DELAY_ENTRE_LOTES` MAIOR; RESPOSTA COM MENOS ITENS DO QUE O LOTE PEDE `TAMANHO_LOTE` MENOR OU `MAX_TOKENS_RESPOSTA` MAIOR. NOS DOIS CASOS, RODE DE NOVO: O QUE JA FOI RESOLVIDO NAO SERA REENVIADO.

In [6]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
    resolvidos_ia = 0
    falhados_ia = 0
elif not novos:
    print("NENHUM PAR INEDITO. NADA A ENVIAR A IA.")
    resolvidos_ia = 0
    falhados_ia = 0
else:
    print(f"ENVIANDO {len(novos)} PARES AO MODELO {modelo}...")
    print()
    resolvidos_ia, falhados_ia = padronizar_novos(conn, client, modelo, novos)
    print()
    print(f"RESOLVIDOS PELA IA        : {resolvidos_ia}")
    print(f"PENDENTES POR FALHA       : {falhados_ia}")

ENVIANDO 5249 PARES AO MODELO deepseek-v4-flash...

LOTE 175/175  |  RESOLVIDOS: 5249  |  PENDENTES POR FALHA: 0

RESOLVIDOS PELA IA        : 5249
PENDENTES POR FALHA       : 0


## 7. PASSO 5: ESPALHAR O RESULTADO PARA AS LINHAS

ATE AQUI, O RESULTADO ESTA NO DICIONARIO, UMA LINHA POR PAR DISTINTO. AGORA ELE E COPIADO PARA TODAS AS LINHAS DE ACIDENTE QUE TEM AQUELE PAR EXATO.

UM PAR RESOLVIDO VIRA `revisao_status = 'ok'`. ISSO VALE TAMBEM PARA OS PARES QUE A IA MARCOU COMO SEM ENDERECO APROVEITAVEL: A REVISAO DELES ESTA CONCLUIDA, SO QUE O RESULTADO E VAZIO.

ESSA DISTINCAO E O QUE PERMITE A COBERTURA FECHAR EM `ok` E A PROXIMA EXECUCAO NAO REENVIAR NADA.

In [7]:
if RECORTE_JA_CONCLUIDO:
    print("ETAPA PULADA: O RECORTE JA ESTA CONCLUIDO.")
else:
    # COPIA A VERSAO PADRONIZADA PARA TODAS AS LINHAS DE CADA PAR.
    linhas_ok, linhas_sem_endereco = espalhar_resultado(conn, modelo, ano, ibge)

    print(f"LINHAS MARCADAS COMO 'ok'          : {linhas_ok}")
    print(f"   DESSAS, SEM ENDERECO APROVEITAVEL: {linhas_sem_endereco}")
    print()
    print("AS LINHAS SEM ENDERECO APROVEITAVEL ESTAO REVISADAS, MAS COM O CAMPO")
    print("PADRONIZADO VAZIO. O NOTEBOOK 04 VAI MARCA-LAS COMO 'sem_via' SEM")
    print("GASTAR NENHUMA BUSCA NEM CHAMADA DE IA COM ELAS.")

LINHAS MARCADAS COMO 'ok'          : 8865
   DESSAS, SEM ENDERECO APROVEITAVEL: 280

AS LINHAS SEM ENDERECO APROVEITAVEL ESTAO REVISADAS, MAS COM O CAMPO
PADRONIZADO VAZIO. O NOTEBOOK 04 VAI MARCA-LAS COMO 'sem_via' SEM
GASTAR NENHUMA BUSCA NEM CHAMADA DE IA COM ELAS.


## 8. PASSO 6: REGISTRAR A COBERTURA E CONFERIR

A COBERTURA E O QUE PERMITE PARAR NO MEIO E CONTINUAR DEPOIS.

O STATUS SO VIRA `ok` QUANDO **NENHUMA** LINHA DO RECORTE FICOU PENDENTE. SE SOBRAR ALGUMA, O STATUS FICA `parcial` E BASTA REEXECUTAR O NOTEBOOK: SO O QUE FALTA SERA PROCESSADO.

A TABELA DE AMOSTRA NO FIM MOSTRA O ANTES E O DEPOIS DE ALGUNS ENDERECOS. E A MELHOR MANEIRA DE JULGAR SE A INSTRUCAO DA ETAPA 2 ESTA FAZENDO O QUE VOCE QUER.

In [8]:
if not RECORTE_JA_CONCLUIDO:
    # REGISTRA O PROGRESSO DESTE RECORTE.
    resumo_cobertura = registrar_cobertura(conn, ano, ibge, len(pares), len(novos))

    print("COBERTURA DESTE RECORTE:")
    for campo, valor in resumo_cobertura.items():
        print(f"   {campo:16}: {valor}")
    print()

    if resumo_cobertura["status"] == "ok":
        print("RECORTE CONCLUIDO. O NOTEBOOK 04 JA PODE PROCESSAR ESTE MUNICIPIO.")
    else:
        print(f"AINDA HA {resumo_cobertura['pendentes']} LINHAS PENDENTES.")
        print("REEXECUTE ESTE NOTEBOOK PARA CONTINUAR DE ONDE PAROU.")

# AMOSTRA DO ANTES E DEPOIS, PARA JULGAR A QUALIDADE DA PADRONIZACAO.
print()
print("AMOSTRA DO ANTES E DEPOIS:")
print()

amostra = pd.read_sql_query(
    f"SELECT {config.COLUNA_ENDERECO} AS end_original, "
    f"       end_acidente_padronizado AS end_padronizado, "
    f"       {config.COLUNA_BAIRRO} AS bairro_original, "
    f"       bairro_acidente_padronizado AS bairro_padronizado "
    f"FROM acidentes_revisado "
    f"WHERE {config.COLUNA_ANO}=? AND {config.COLUNA_MUNICIPIO}=? AND revisao_status='ok' "
    f"LIMIT 15",
    conn, params=(ano, ibge),
)
print(amostra.to_string(index=False))

conn.close()

COBERTURA DESTE RECORTE:
   total           : 8865
   distintos       : 5249
   novos_ia        : 5249
   reaproveitados  : 0
   pendentes       : 0
   status          : ok

RECORTE CONCLUIDO. O NOTEBOOK 04 JA PODE PROCESSAR ESTE MUNICIPIO.

AMOSTRA DO ANTES E DEPOIS:

                           end_original                    end_padronizado              bairro_original           bairro_padronizado
      ALAMEDA DO ALMEIDA  CEP 74915-020                 Alameda do Almeida             SETOR JARDIM LUZ             Setor Jardim Luz
                RUA J 32  CEP 74950-010                           Rua J-32              MANSOES PARAISO              Mansoes Paraiso
                 RUA R 4  CEP 74960-610                            Rua R-4            PARQUE IBIRAPUERA            Parque Ibirapuera
             RUA TAPAJOS  CEP 74905-700                        Rua Tapajos                VILA BRASILIA                Vila Brasilia
RUA JOSE CANDIDO QUEIROZ  CEP 74980-070           Rua Jose Candid

## 9. VISAO GERAL DE TODOS OS RECORTES

ESTA CELULA PODE SER EXECUTADA A QUALQUER MOMENTO, INDEPENDENTE DO RESTO DO NOTEBOOK.

ELA MOSTRA O MAPA COMPLETO DO QUE JA FOI REVISADO, POR ANO E MUNICIPIO. E DAQUI QUE VOCE SABE O QUE FALTA.

In [9]:
conexao = sqlite3.connect(str(config.BANCO))

try:
    cobertura_geral = pd.read_sql_query(
        "SELECT ano, codigo_ibge, status, total_linhas, distintos, novos_ia, "
        "       reaproveitados, concluido_em "
        "FROM revisao_cobertura ORDER BY ano DESC, codigo_ibge",
        conexao,
    )

    if cobertura_geral.empty:
        print("NENHUM RECORTE REVISADO AINDA.")
    else:
        print("COBERTURA DA REVISAO:")
        print()
        print(cobertura_geral.to_string(index=False))

        # A ECONOMIA ACUMULADA DO DICIONARIO, SOMANDO TODOS OS RECORTES.
        total_distintos = int(cobertura_geral["distintos"].sum())
        total_ia = int(cobertura_geral["novos_ia"].sum())

        if total_distintos:
            print()
            print(f"PARES DISTINTOS PROCESSADOS : {total_distintos}")
            print(f"ENVIADOS A IA               : {total_ia}")
            print(f"ECONOMIA PELO DICIONARIO    : {(1 - total_ia / total_distintos) * 100:.1f}%")
finally:
    conexao.close()

COBERTURA DA REVISAO:

 ano codigo_ibge status  total_linhas  distintos  novos_ia  reaproveitados                     concluido_em
2024     5201405     ok          8865       5249      5249               0 2026-08-13T17:17:26.691020+00:00

PARES DISTINTOS PROCESSADOS : 5249
ENVIADOS A IA               : 5249
ECONOMIA PELO DICIONARIO    : 0.0%


## PROXIMO PASSO

COM O RECORTE MARCADO COMO `ok`, O NOTEBOOK **04_associacao_vias.ipynb** PODE VINCULAR ESSES ACIDENTES AS VIAS.

ELE TAMBEM PRECISA DA TABELA `vias_processadas`, ENTAO CONFIRA QUE O NOTEBOOK **02** JA FOI RODADO.